# Eva Data

This notebook builds a `MultiDataset` containing exactly one `ZarrDataset`, loads one batch, visualizes one image with `mediapy`, and prints the rest of the batch.

In [ ]:
from pathlib import Path

import cv2
import imageio_ffmpeg
import mediapy as mpy
import numpy as np
import torch

from egomimic.rldb.embodiment.eva import Eva
from egomimic.rldb.embodiment.human import Aria
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, ZarrDataset
from egomimic.rldb.zarr.zarr_dataset_multi import S3EpisodeResolver
from egomimic.utils.egomimicUtils import (
    INTRINSICS,
    cam_frame_to_cam_pixels,
    nds,
)
from egomimic.utils.aws.aws_data_utils import load_env

# Ensure mediapy can find an ffmpeg executable in this environment
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

In [ ]:
TEMP_DIR = "/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest"
load_env()

In [ ]:
# Point this at a single episode directory, e.g. /path/to/episode_hash.zarr
# EPISODE_PATH = Path("/coc/flash7/scratch/egoverseDebugDatasets/1767495035712.zarr")

key_map = Eva.get_keymap()
transform_list = Eva.get_transform_list()

# Build a MultiDataset with exactly one ZarrDataset inside
# single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map, transform_list=transform_list)
# single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map)

# multi_ds = MultiDataset(datasets={"single_episode": single_ds}, mode="total")
resolver = S3EpisodeResolver(
    TEMP_DIR, key_map=key_map, transform_list=transform_list
)
filters = {
    "episode_hash": "2025-12-26-18-07-46-296000"
}
multi_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)

In [ ]:
# Separate YPR visualization preview
for batch in loader:
    vis_ypr = Eva.viz_transformed_batch(batch, mode="palm_axes")
    mpy.show_image(vis_ypr)
    break

In [ ]:
images = []
for i, batch in enumerate(loader):
    vis = Eva.viz_transformed_batch(batch, mode="palm_traj")
    images.append(vis)
    if i > 10:
        break

mpy.show_video(images, fps=30)

## Human Datasets
Mecka, Scale and Aria should all run exactly the same

In [ ]:
temp_dir = "/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest"

intrinsics_key = "base"

key_map = Aria.get_keymap()
transform_list = Aria.get_transform_list()

resolver = S3EpisodeResolver(
    temp_dir,
    key_map=key_map,
    transform_list=transform_list,
)

filters = {"episode_hash": "2026-01-20-20-59-43-376000"} #aria
# filters = {"episode_hash": "692ee048ef7557106e6c4b8d"} # mecka

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
ims = []
for i, batch in enumerate(loader):
    vis = Aria.viz_transformed_batch(batch, mode="palm_traj")
    ims.append(vis)
    # mpy.show_image(vis)

    # for k, v in batch.items():
    #     print(f"{k}: {tuple(v.shape)}")
    
    if i > 10:
        break

mpy.show_video(ims, fps=30)


In [ ]:
# Aria YPR video (same data loop, YPR overlay)
ims_ypr = []
for i, batch in enumerate(loader):
    vis_ypr = Aria.viz_transformed_batch(batch, mode="palm_axes")
    ims_ypr.append(vis_ypr)
    if i > 20:
        break

mpy.show_video(ims_ypr, fps=30)

## Keypoint Visualization

In [ ]:
# Load Scale episode with raw keypoints (no action chunking needed)

from egomimic.rldb.zarr.action_chunk_transforms import _xyzwxyz_to_matrix

key_map_kp = {
    "images.front_1": {"zarr_key": "images.front_1"},
    "left.obs_keypoints": {"zarr_key": "left.obs_keypoints"},
    "right.obs_keypoints": {"zarr_key": "right.obs_keypoints"},
    "obs_head_pose": {"zarr_key": "obs_head_pose"},
}

filters = {"episode_hash": "2026-01-20-20-59-43-376000"}

resolver = S3EpisodeResolver(
    temp_dir,
    key_map=key_map
)

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader_kp = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
# ARIA Keypoint Viz
# MANO skeleton edges: (parent, child) for drawing bones
MANO_EDGES = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # index
    (0, 9), (9, 10), (10, 11), (11, 12),    # middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # ring
    (0, 17), (17, 18), (18, 19), (19, 20),  # pinky
]

# aria configuration
MANO_EDGES = [
    (5, 6,), (6, 7), (7, 0), # thumb
    (5, 8), (8, 9), (9, 10), (9, 1), # index
    (5, 11), (11, 12), (12, 13), (13, 2), # middle
    (5, 14), (14, 15), (15, 16), (16, 3), # ring
    (5, 17), (17, 18), (18, 19), (19, 4), # pinky
]

FINGER_COLORS = {
    "thumb": (255, 100, 100),   # red
    "index": (100, 255, 100),   # green
    "middle": (100, 100, 255),  # blue
    "ring": (255, 255, 100),    # yellow
    "pinky": (255, 100, 255),   # magenta
}
FINGER_EDGE_RANGES = [
    ("thumb", 0, 3), ("index", 3, 6), ("middle", 6, 9),
    ("ring", 9, 12), ("pinky", 12, 15),
]


def viz_keypoints(batch, image_key="observations.images.front_img_1"):
    """Visualize all 21 MANO keypoints per hand, projected onto the image."""
    # Prepare image
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()
    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS["base"]
    head_pose = batch["obs_head_pose"][0].detach().cpu().numpy()  # (6,)

    # T_head_world: camera pose in world (camera-to-world)
    # We need world-to-camera = inv(T_head_world)
    T_head_world = _xyzwxyz_to_matrix(head_pose[None, :])[0]  # (4, 4)
    T_world_to_cam = np.linalg.inv(T_head_world)

    vis = img_np.copy()
    h, w = vis.shape[:2]

    for hand, dot_color in [("left", (0, 120, 255)), ("right", (255, 80, 0))]:
        kps_key = f"{hand}.obs_keypoints"
        if kps_key not in batch:
            continue
        kps_flat = batch[kps_key][0].detach().cpu().numpy()  # (63,)
        kps_world = kps_flat.reshape(21, 3)

        # Skip if keypoints are all zero (invalid, clamped from 1e9)
        if np.allclose(kps_world, 0.0, atol=1e-3):
            continue

        # World -> camera frame
        kps_h = np.concatenate([kps_world, np.ones((21, 1))], axis=1)  # (21, 4)
        kps_cam = (T_world_to_cam @ kps_h.T).T[:, :3]  # (21, 3)

        # Camera frame -> pixels
        kps_px = cam_frame_to_cam_pixels(kps_cam, intrinsics)  # (21, 3+)

        # Identify valid keypoints (z > 0 and in image bounds)
        valid = (kps_cam[:, 2] > 0.01)
        valid &= (kps_px[:, 0] >= 0) & (kps_px[:, 0] < w)
        valid &= (kps_px[:, 1] >= 0) & (kps_px[:, 1] < h)

        # Draw skeleton edges (colored by finger)
        for finger, start, end in FINGER_EDGE_RANGES:
            color = FINGER_COLORS[finger]
            for edge_idx in range(start, end):
                i, j = MANO_EDGES[edge_idx]
                if valid[i] and valid[j]:
                    p1 = (int(kps_px[i, 0]), int(kps_px[i, 1]))
                    p2 = (int(kps_px[j, 0]), int(kps_px[j, 1]))
                    cv2.line(vis, p1, p2, color, 2)

        # Draw keypoint dots on top
        for k in range(21):
            if valid[k]:
                center = (int(kps_px[k, 0]), int(kps_px[k, 1]))
                cv2.circle(vis, center, 4, dot_color, -1)
                cv2.circle(vis, center, 4, (255, 255, 255), 1)  # white border

        # Label wrist
        if valid[0]:
            wrist_px = (int(kps_px[0, 0]) + 6, int(kps_px[0, 1]) - 6)
            cv2.putText(vis, f"{hand[0].upper()}", wrist_px,
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, dot_color, 2)

    return vis

In [ ]:
# Render keypoint video
ims_kp = []
for i, batch_kp in enumerate(loader_kp):
    vis = viz_keypoints(batch_kp)
    ims_kp.append(vis)
    if i > 10:
        break

mpy.show_video(ims_kp, fps=30)